# CP2 Week 5 -- Matplotlib Standards

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-4
**Focus:** professional plots, labels, legends, saving figures

## Learning Objectives
- Create professional-quality plots with matplotlib
- Always include titles, axis labels, legends, and grids
- Save figures programmatically at publication quality
- Build standard plotting functions for your pipeline
- Know when to use different plot types

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: The 5 Rules of Professional Plots

Every plot you create in your v2 pipeline MUST follow these rules:

1. **Title** -- what does this plot show?
2. **Axis labels** -- with units (e.g., "Temperature (C)")
3. **Grid** -- for readability
4. **Legend** -- if there are multiple series
5. **Save at high DPI** -- 150+ for reports

A plot without labels is like a table without headers -- useless.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for Colab/scripts
import matplotlib.pyplot as plt
import os

# === GOOD plot: follows all 5 rules ===
import random
random.seed(42)
temps = [20 + random.gauss(0, 3) for _ in range(50)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(temps)), temps, linewidth=1.0, color="#2196F3")

ax.set_title("Sensor Temperature Over Time", fontsize=14, fontweight="bold")  # Rule 1
ax.set_xlabel("Reading Number", fontsize=12)        # Rule 2
ax.set_ylabel("Temperature (C)", fontsize=12)       # Rule 2
ax.grid(True, alpha=0.3)                            # Rule 3
# No legend needed (single series)                  # Rule 4

os.makedirs("reports/figures", exist_ok=True)
fig.savefig("reports/figures/demo_good.png", dpi=150, bbox_inches="tight")  # Rule 5
plt.close(fig)
print("Saved: reports/figures/demo_good.png")

**Expected Output:**
```
Saved: reports/figures/demo_good.png
```

### Standard Time Series Function

In [ ]:
def standard_timeseries(values, title, ylabel, savepath, threshold=None):
    """Create a standardized time series plot.
    
    Args:
        values: list of numbers
        title: plot title
        ylabel: y-axis label (include units!)
        savepath: where to save the figure
        threshold: optional horizontal threshold line
    
    Returns:
        matplotlib Figure object
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Main data line
    ax.plot(range(len(values)), values,
            linewidth=1.0, color="#2196F3", label="Data")
    
    # Mean line
    mean_val = sum(values) / len(values)
    ax.axhline(y=mean_val, color="#4CAF50", linestyle=":",
               linewidth=1.0, label="Mean (" + str(round(mean_val, 1)) + ")")
    
    # Threshold line (optional)
    if threshold is not None:
        ax.axhline(y=threshold, color="#F44336", linestyle="--",
                   linewidth=1.5, label="Threshold (" + str(threshold) + ")")
    
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Time Index", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    
    os.makedirs(os.path.dirname(savepath), exist_ok=True)
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)
    return fig

# Demo
values = [50 + random.gauss(0, 10) for _ in range(100)]
standard_timeseries(values, "Temperature Over Time",
                    "Temperature (C)",
                    "reports/figures/timeseries.png",
                    threshold=65)

**Expected Output:**
```
Saved: reports/figures/timeseries.png
```

### Standard Summary Plot (Histogram + Box Plot)

In [ ]:
def standard_summary(values, title, xlabel, savepath):
    """Create histogram + box plot side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                    gridspec_kw={"width_ratios": [3, 1]})
    
    # Histogram
    n_bins = min(30, len(values) // 5 + 1)
    ax1.hist(values, bins=n_bins, color="#2196F3",
             edgecolor="white", alpha=0.8)
    
    mean_val = sum(values) / len(values)
    ax1.axvline(mean_val, color="#F44336", linestyle="-",
                linewidth=2, label="Mean (" + str(round(mean_val, 1)) + ")")
    
    sorted_v = sorted(values)
    median_val = sorted_v[len(values) // 2]
    ax1.axvline(median_val, color="#FF9800", linestyle="--",
                linewidth=2, label="Median (" + str(round(median_val, 1)) + ")")
    
    ax1.set_title(title + " -- Distribution", fontsize=14, fontweight="bold")
    ax1.set_xlabel(xlabel, fontsize=12)
    ax1.set_ylabel("Count", fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    ax2.boxplot(values, vert=True)
    ax2.set_title("Box Plot", fontsize=12)
    ax2.set_ylabel(xlabel, fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)
    return fig

standard_summary(values, "Temperature", "Degrees C",
                 "reports/figures/summary.png")

**Expected Output:**
```
Saved: reports/figures/summary.png
```

---
## Part 3: Bar Charts for Category Counts

Bar charts are perfect for showing how many items are in each category.

In [ ]:
def standard_barchart(categories, values, title, ylabel, savepath):
    """Create a standardized bar chart."""
    fig, ax = plt.subplots(figsize=(10, 5))
    
    colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]
    bar_colors = [colors[i % len(colors)] for i in range(len(categories))]
    
    ax.bar(categories, values, color=bar_colors, edgecolor="white")
    
    # Add value labels on top of each bar
    for i, (cat, val) in enumerate(zip(categories, values)):
        ax.text(i, val + max(values) * 0.02, str(val),
                ha="center", va="bottom", fontweight="bold")
    
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.3, axis="y")
    
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)
    return fig

standard_barchart(
    ["Missing", "Non-numeric", "Out of range", "Kept"],
    [15, 8, 12, 165],
    "Cleaning Results by Category",
    "Number of Rows",
    "reports/figures/cleaning_bar.png"
)

**Expected Output:**
```
Saved: reports/figures/cleaning_bar.png
```

---
## Part 4: Scatter Plots

Scatter plots show the relationship between two numeric variables.

In [ ]:
def standard_scatter(x_vals, y_vals, title, xlabel, ylabel, savepath):
    """Create a standardized scatter plot."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    ax.scatter(x_vals, y_vals, alpha=0.6, color="#2196F3",
               edgecolors="white", s=50)
    
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.3)
    
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)
    return fig

# Demo: value vs. score
import random
random.seed(42)
x = [random.gauss(50, 15) for _ in range(80)]
y = [v * 0.8 + random.gauss(0, 10) for v in x]  # correlated
standard_scatter(x, y, "Value vs. Score", "Value", "Score",
                 "reports/figures/scatter.png")

**Expected Output:**
```
Saved: reports/figures/scatter.png
```

---
## Part 5: Multi-Panel Figures

For reports, combine multiple plots into one figure.

In [ ]:
def create_dashboard(values, categories, cat_counts, savepath):
    """Create a 2x2 dashboard figure."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Top-left: time series
    axes[0, 0].plot(range(len(values)), values, color="#2196F3", linewidth=0.8)
    axes[0, 0].set_title("Time Series", fontweight="bold")
    axes[0, 0].set_xlabel("Index")
    axes[0, 0].grid(True, alpha=0.3)
    
    # Top-right: histogram
    axes[0, 1].hist(values, bins=20, color="#4CAF50", edgecolor="white")
    axes[0, 1].set_title("Distribution", fontweight="bold")
    axes[0, 1].set_xlabel("Value")
    axes[0, 1].grid(True, alpha=0.3)
    
    # Bottom-left: bar chart
    axes[1, 0].bar(categories, cat_counts, color="#FF9800")
    axes[1, 0].set_title("Category Counts", fontweight="bold")
    axes[1, 0].grid(True, alpha=0.3, axis="y")
    
    # Bottom-right: box plot
    axes[1, 1].boxplot(values)
    axes[1, 1].set_title("Box Plot", fontweight="bold")
    axes[1, 1].grid(True, alpha=0.3)
    
    fig.suptitle("Data Analysis Dashboard", fontsize=16, fontweight="bold")
    plt.tight_layout()
    fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: " + savepath)

create_dashboard(
    values,
    ["Cat A", "Cat B", "Cat C"],
    [45, 35, 20],
    "reports/figures/dashboard.png"
)

**Expected Output:**
```
Saved: reports/figures/dashboard.png
```

### Try It Yourself

Create a figure with your project data. Choose the plot types that make the most sense for your data.

In [ ]:
# Try it: create a plot for your project
# TODO: choose plot type and create it


### Common Mistakes: Matplotlib

**Mistake 1:** Forgetting plt.close(fig) -- memory leaks if you create many figures.

**Fix:** Always call plt.close(fig) after saving.

**Mistake 2:** Not using bbox_inches='tight' -- labels get cut off.

**Fix:** Always add bbox_inches='tight' to savefig.

**Mistake 3:** Hard-coding file paths -- breaks on different machines.

**Fix:** Use os.path.join and os.makedirs to build paths.

**Mistake 4:** Using plt.show() in a script -- blocks execution.

**Fix:** Use matplotlib.use('Agg') and plt.close(fig) instead.


### Debugging Tips: Matplotlib

- If figures are blank, check that you are plotting to the right axes object
- If labels are cut off, add bbox_inches='tight' to savefig
- If colors look wrong, use hex codes like '#2196F3' for consistency
- If the figure file is 0 bytes, check that plt.close was not called before savefig

### Key Takeaway

- Every plot MUST have: title, axis labels, grid, legend (if multiple series)
- Save at 150+ DPI for report quality
- Always call plt.close(fig) after saving to free memory
- Use standard functions so all plots look consistent
- Multi-panel figures are great for dashboards

---
## Mini-Quiz

In [ ]:
# Q1: What are the 5 rules of professional plots?
# Answer: 

# Q2: Why use matplotlib.use("Agg")?
# Answer: 

# Q3: What DPI should you use for report figures?
# Answer: 

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Create a time series plot of your project data.


In [ ]:
# HW2: Create a histogram of your main numeric column.


In [ ]:
# HW3: Add threshold line and mean to your plot.


In [ ]:
# HW4: Save both figures to reports/figures/.


### Practice (5-8)

In [ ]:
# HW5: Create a bar chart of category counts.


In [ ]:
# HW6: Create a scatter plot of two numeric columns.


In [ ]:
# HW7: Create a multi-panel figure (2x2 grid of plots).


In [ ]:
# HW8: Write a plot function that automatically labels axes
# based on column names in the config.


### Challenge (9-11)

In [ ]:
# HW9: Create a "before/after cleaning" comparison plot.


In [ ]:
# HW10: Add confidence bands (mean +/- std) to timeseries.


In [ ]:
# HW11: Create a custom color scheme for your track.


### Mini-Project

In [ ]:
# HW12: Build a complete plotting module for your project.
# Requirements: 3+ plot types, all saved to files, professional quality.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)